# The ReAct Loop: The Model Emits Text, Your Code Acts

You have probably already used an agent. You may not have called it that.

Think of the last time you asked an AI chat to do something with more than one step. Maybe you asked it to compare three products and recommend one, and you watched it work: *Searching the web. Reading a product page. Checking another source.* Then it handed you a recommendation, with reasons. Or you asked it to plan a trip, look a few things up, do some math, and write up the result. Little status lines appeared while it worked, and a finished answer came out the other side.

It felt like the AI went off and did things for you, the way an assistant would. Browsing. Reading. Comparing. Calculating. Working on your behalf while you watched.

Here is the part nobody puts on the box: **the AI did none of those things.**

The model behind that chat cannot browse. It cannot open a page, run a calculation, send a message, or save a file. It has no hands. What it can do, the only thing it can do, is produce text, one reply at a time.

Every action you watched happen was ordinary code, written by programmers, sitting next to the model. That code knew how to search the web, open pages, and do arithmetic. The model's entire contribution was to write the right words at the right moment: *search for this. Open that page. Add these two numbers.* The code read the words, did the real work, and handed the result back to the model as more words. Then the model wrote the next instruction. Back and forth, model and code, many times a minute, until the model finally wrote the answer.

What you used was an agent: a model that only talks, wrapped in code that listens and acts. This notebook takes that wrapping apart, piece by piece, and then has you build a small one yourself.

**The one idea:** An agent is a loop, and the model never *does* anything. It emits text. Your code acts on that text. The result goes back in as more text. By the end of this notebook you will have seen the whole trick.

*Week 1 of the agent sprint, as a lesson. The build was the exercise; this is the concept behind it.*

**What Week 1 is for:** experiencing the loop, end to end, with your own hands. It is *not* about getting an agent to work perfectly. A small, imperfect, occasionally misbehaving agent that you have watched think is a successful Week 1. If you catch yourself spending hours trying to make the agent behave perfectly, stop: that is not the assignment. That trap is real, and it eats weeks.

**How to read this notebook.** Every new idea is introduced the same way, in the same order:

1. **A definition** — what the thing is, in plain words.
2. **An overview** — what it is for, and where it fits in the whole machine.
3. **An example** — the thing running, on real sample input.
4. **The mechanics** — what it is made of, including the code.

Before every code cell, a short lead-in tells you the cell's purpose, what it does, and what to expect when you run it, including why that output matters. The comment block at the top of each code cell is that lead-in, compressed. Read the lead-in, run the cell, check the output against what the lead-in promised. That check is your checkpoint before the next piece builds on this one.

## The Whole Thing, Once, Slowly: A Toy Walkthrough

Before any code, here is the entire process, start to finish, on the simplest question we have. Nothing in the rest of this notebook is new after this section. The rest is these same steps, taken one at a time.

**You type a question.** Just the question, nothing else:

> What is the capital of France?

**Your code wraps the question before the model ever sees it.** When you hit send, the *contract* is placed in front of your question. The contract is a set of instructions telling the model how to respond: what shape its reply must take, and what tools it may ask for. Here is a shortened version of what actually gets sent:

```
You solve problems by reasoning step by step, then taking actions.
Reply using exactly the format below, nothing else.

Thought: <your reasoning>
Action: <one of: calculator, search>
Action Input: <input for the action>
... (repeat Thought / Action / Action Input as needed)
Thought: <final reasoning>
Final Answer: <the answer>

Question: What is the capital of France?
```

*(Shortened here so you can see the whole thing at a glance. The real contract, with a worked example baked into it, is built in the Contract section below.)*

> **Remember this, because everything follows from it:** the model only ever responds to the prompt that is sent to it. That is all it does. It cannot see your code, your files, or your intentions. It does not know what a search engine is. It knows the text above, and nothing else in the world.

**The model replies, in text.** It follows the contract and writes:

```
Thought: The question asks for the capital of France. That is a fact to look up, so I will use the search tool.
Action: search
Action Input: capital of France
```

**Now your code does the work.** Notice that the model did not search for anything. It wrote the *word* "search." Your code takes over:

1. The **parser** scans the reply and pulls out the two facts: the tool name (`search`) and its input (`capital of France`).
2. Your code looks the name up in the **registry**, the list of tools, and finds the search function *you* wrote.
3. Your code runs that function. The result: "The capital of France is Paris."
4. Your code glues that result onto the conversation as a new line: `Observation: The capital of France is Paris.`

**The whole updated prompt goes back to the model.** Not just the new part. *All of it, every time:* the contract again, the question again, plus the model's own reply and the Observation, glued on at the bottom. Here it is, exactly as the model receives it on turn 2, with each piece labeled:

```
----- 1. THE CONTRACT (the instructions, sent in front, every time) -----
You solve problems by reasoning step by step, then taking actions.
Reply using exactly the format below, nothing else.

Thought: <your reasoning>
Action: <one of: calculator, search>
Action Input: <input for the action>
... (repeat Thought / Action / Action Input as needed)
Thought: <final reasoning>
Final Answer: <the answer>

----- 2. THE QUESTION (what you typed) -----
Question: What is the capital of France?

----- 3. THE MODEL'S FIRST REPLY (new: appended after turn 1) -----
Thought: The question asks for the capital of France. That is a fact to look up, so I will use the search tool.
Action: search
Action Input: capital of France

----- 4. THE OBSERVATION (new: your code's answer, appended last) -----
Observation: The capital of France is Paris.
```

**The model reads this whole thing, top to bottom, and replies again.** It has no memory of turn 1 beyond what this text shows it. Everything it knows about what has happened so far is sitting in sections 3 and 4:

```
Thought: The search result gave me the answer.
Final Answer: Paris
```


**Your code sees `Final Answer:` and stops.** It hands "Paris" back to you. Done. Two trips around the loop.

**So what is "the agent," exactly?** Not the model. Think of the agent as everything wrapped *around* the model: the instructions (the contract), the coded functions (the parser, the tools, the registry), and the error handling that catches a bad reply or a failed tool and turns it into more text the model can react to. The model supplies words. The wrapping turns words into actions, and actions into more words, until the model says it is finished. That wrapping is what you build this week.

**How this maps to production:** it does not change. A production agent is this same loop with the toy parts swapped for real ones: a real API call instead of a stand-in model, a real search engine instead of a three-fact index, real tools with real consequences, which is why the error handling matters there even more than it does here. The shape you are about to learn *is* the production shape.

**And one expectation to set now:** in this walkthrough the model behaved perfectly. Real models do not, and yours will not either: it will break the format, ask for tools that do not exist, and occasionally talk itself in circles. That is normal, and for Week 1 it is even useful, because watching the loop recover is how you learn what the loop is. Week 1 is about *experiencing* the loop, not about getting the agent to work perfectly. Perfection is a later week's problem.

## Anatomy of a Turn

**Definition.** A *turn* is one trip around the loop: the model writes one reply, your code reacts to it, and the result goes back to the model. An agent is just turns, repeated until the job is done.

Every turn has the same four lines:

- **Thought:** the model's private reasoning, in plain words.
- **Action:** the name of a tool it wants to use.
- **Action Input:** the argument to hand that tool.
- **Observation:** what your code feeds back after running the tool. The model never writes this line. You do.

![The ReAct loop](react-loop-diagram.png)

Notice what the model never touches: the calculator, the search index, your filesystem. It writes the *word* "calculator" and your code does the rest. That separation is the entire safety story of Week 1.

### Setup: three libraries this notebook borrows

**Purpose of the cell below:** load the three Python libraries the rest of the notebook uses, before anything needs them.

**What to expect:** nothing. This cell prints nothing and that is correct. It just makes three tools available:

- `re` — pattern matching. This is how the parser finds the `Action:` line inside a model's reply.
- `ast` — reads arithmetic as a structure we can inspect, so the calculator can do math *without* ever running raw code (no `eval()`).
- `operator` — the actual +, -, *, / functions the calculator is allowed to use.

**Why it matters:** every later cell assumes these are loaded. If a later cell ever complains about a missing name, this cell did not run.

In [1]:
# Setup: load the three libraries the notebook uses.
# re = find shaped text (the parser). ast = inspect math safely (the calculator).
# operator = the handful of math operations the calculator is allowed to perform.
import re        # pattern-matching: how we find the "Action:" line in model text
import ast       # safe math: lets us read arithmetic as a tree structure without ever running eval()
import operator  # the actual +, -, *, / functions the calculator is allowed to use

## The Contract: A Prompt and a Parser

Three definitions first, because this whole section is built from them:

- **Prompt.** The text you send the model. All of it: the instructions, the question, and (as you will see later) everything that has happened so far. The prompt is the model's entire world. It sees nothing else.
- **Parser.** A small piece of your code that reads a text reply and pulls out the parts you need. Think of it as a very literal assistant who scans a paragraph and hands you back two facts from it.
- **Contract.** An agreement about shape. The prompt is your side of the contract: "always reply in exactly this format." The parser is the enforcement side: "I will read replies that follow the format, and I will notice when one does not."

**The overview.** The model does not know it is an agent. It has no buttons to press and no way to reach out and run anything. One piece of vocabulary to settle right now, because everyone uses it: you will constantly hear that "the model calls a tool," and this notebook will say it too. It is shorthand. What literally happens is smaller: the model writes the *name* of a tool in its reply, as text. Then your code does the actual calling: it runs the function and feeds the result back to the model as more text. The model requests. Your code calls. Hold onto that split.

So: all the model can do is produce text. We give it a contract written in plain English: reply in exactly the Thought / Action / Action Input format, with one worked example inside the prompt showing the shape. Then our code holds up its end with the parser, which pulls the action name and its input out of each reply so your code can act on them.


Formatting discipline plus a parser. That is the whole "agentic" trick at this level.

**The cell below** builds that contract as a piece of text (called `REACT_PROMPT`) plus one small function, `build_prompt`, that slots a new question into it. Run it and expect to see the full contract printed: the format instructions, the worked example with the two calculator steps, and at the very bottom, the question you passed in, sitting where the model will read it last. This printout is the checkpoint: if the printed contract looks wrong, everything downstream is wrong, because this text is all the model ever knows.

In [2]:
# REACT_PROMPT: the contract, written in plain English.
# It tells the model the exact reply format (Thought / Action / Action Input /
# Final Answer) and shows one worked example so the shape is unambiguous.
# build_prompt() slots a new question into the template.
# If the model drifts from the format, fix the prompt first, the parser second.
REACT_PROMPT = """You solve problems by reasoning step by step, then taking actions.
Reply using exactly the format below, nothing else.

Thought: <your reasoning>
Action: <one of: calculator, search>
Action Input: <input for the action>
... (repeat Thought/Action/Action Input as needed)
Thought: <final reasoning>
Final Answer: <the answer>

Example:
Question: What is 12*34 plus 8?
Thought: I need to compute 12*34 first.
Action: calculator
Action Input: 12*34
Observation: 408
Thought: Now add 8.
Action: calculator
Action Input: 408+8
Observation: 416
Thought: I have the answer.
Final Answer: 416

Question: {question}
"""

def build_prompt(question):
    return REACT_PROMPT.format(question=question)

print(build_prompt("What is 12*34 plus 8?"))

You solve problems by reasoning step by step, then taking actions.
Reply using exactly the format below, nothing else.

Thought: <your reasoning>
Action: <one of: calculator, search>
Action Input: <input for the action>
... (repeat Thought/Action/Action Input as needed)
Thought: <final reasoning>
Final Answer: <the answer>

Example:
Question: What is 12*34 plus 8?
Thought: I need to compute 12*34 first.
Action: calculator
Action Input: 12*34
Observation: 408
Thought: Now add 8.
Action: calculator
Action Input: 408+8
Observation: 416
Thought: I have the answer.
Final Answer: 416

Question: What is 12*34 plus 8?



### First, what is a regex?

The contract above promised a parser that "pulls out" the action name and its input. The pulling is done with **regexes**, so before the parser makes sense, this does.

**Definition.** A regex (short for *regular expression*) is a search pattern that describes the *shape* of text instead of the exact words. An ordinary search says: find the word "calculator." That only works if you already know the word. A regex search says: find the line that starts with `Action:` and hand me back whatever word comes after the colon. That works no matter which tool name the model wrote.

Here is a real model reply, the same kind the parser will eat:

```
Thought: I need to multiply 12 by 34, then add 8.
Action: calculator
Action Input: 12*34 + 8
```

And here is the first regex. This exact pattern appears inside the parser in the next section:

```
Action:\s*(\w+)
```

Read it left to right, piece by piece:

- `Action:` — find this exact text.
- `\s*` — then skip any spaces. (`\s` means a whitespace character; `*` means "zero or more of them," so sloppy spacing is forgiven.)
- `(\w+)` — then take one or more "word characters" (letters, digits, underscore). The parentheses are the most important part of the whole pattern: they mark **the piece you want handed back to you**.

Run that pattern on the reply above and it finds `Action:` and hands back **calculator**.

The second regex is its sibling:

```
Action Input:\s*(.+)
```

Same start, but `.` means "any character at all," so `.+` means "everything from here to the end of the line." It hands back **12*34 + 8**.

That is all a regex is: a standing instruction for finding shaped text, with parentheses around the part you keep. The parser is just these two searches, one after the other.

## parse_action: Trust, but Verify

**Definition.** `parse_action` is the parser from the contract section: one small function that takes a model's whole reply as text and pulls out the two facts your code needs — which tool the model asked for, and what to hand that tool.

**What it promises.** It returns the pair `(name, input)` when it finds an `Action:` line. When it cannot find one, it returns a special Python value called `None`, which you can read as "nothing." `None` is **not a crash**. It is a signal: "this reply did not follow the contract." The loop, later, uses that signal to send the model a correction instead of giving up.

**Run the cell below to watch the parser work on three test replies.** Each one represents a kind of reply a real model actually sends. Here is what each is, and what to expect:

1. **`good`** — a clean reply that follows the contract exactly: a Thought, `Action: calculator`, `Action Input: 12*34`. *Expect:* `('calculator', '12*34')` — the two pieces, pulled out. This is the normal case.
2. **`no_action`** — a reply where the model ignored the contract and just answered in plain words: "I think the answer is 42." There is no `Action:` line anywhere in it. *Expect:* `None`. The parser found no action to pull out, and says so without crashing. This is the case that keeps the loop alive when a model misbehaves.
3. **`fenced`** — a good reply wrapped in markdown code fences (the triple-backtick wrapping models love to add when they are "helpfully" formatting). *Expect:* `('search', 'cats')`. Before searching, the parser peels the fences off, then reads the reply exactly like the clean case.

**Why this output is your checkpoint:** these three lines of output are the whole contract, verified. Clean case works, broken case signals instead of crashing, dressed-up case still reads correctly. Every later cell trusts the parser. This is where you check that trust is earned.

In [3]:
# parse_action: the parser half of the contract.
# The prompt tells the model "reply in this exact format"; this function checks
# whether it actually did. Two small regexes: one finds "Action: <name>", the other
# finds "Action Input: <text>". No Action line -> returns None, which the loop
# treats as "the model broke format, send it a correction" (not a crash).
def parse_action(reply):
    """Pull (action_name, action_input) from a model reply.
    Returns None when no Action: line is found -- that is a signal, not a crash."""
    text = reply.strip()
    text = re.sub(r'^```[a-z]*\n?', '', text)   # strip opening fence
    text = re.sub(r'\n?```$', '', text)          # strip closing fence
    m_action = re.search(r'Action:\s*(\w+)', text)
    if not m_action:
        return None
    m_input = re.search(r'Action Input:\s*(.+)', text)
    return m_action.group(1), (m_input.group(1).strip() if m_input else '')

good = "Thought: I need 12*34.\nAction: calculator\nAction Input: 12*34"
no_action = "I think the answer is 42"
fenced = "```\nThought: hi\nAction: search\nAction Input: cats\n```"

print(parse_action(good))       # ('calculator', '12*34')
print(parse_action(no_action))  # None
print(parse_action(fenced))    # ('search', 'cats')

('calculator', '12*34')
None
('search', 'cats')


## Tools: The Things Your Code Can Do

Two definitions first, because the rest of this notebook leans on both words:

- **Tool.** A plain function that *you* wrote, which does one job. The calculator does arithmetic. The search looks facts up. A tool is ordinary code with an ordinary name. The model cannot run a tool. It can only *ask* for one, by writing the tool's name in its reply.
- **Registry.** The list of tools the agent is allowed to use. One list, kept in one place, that pairs each tool's name with the actual function: the name `"calculator"` points at the `calculator` function, the name `"search"` points at the `search` function. If a tool is not on this list, the agent does not have it, full stop.

<div align="center">

**Where the registry fits in a turn.** Watch the relay, with the capital-of-France turn from the walkthrough running underneath it:

The model writes a reply

`Thought: The question asks for the capital of France. That is a fact to look up, so I will use the search tool.`

`Action: search`

`Action Input: capital of France`

↓

The parser pulls out the tool's *name* (just a word) from the `Action:` line, and the tool *input* form the `Action Input:` line

`('search', 'capital of France')`

↓

Your code looks that name up in the registry

`TOOLS["search"]`

↓

The registry hands back the real function

the `search` function itself

↓

Your code runs it with the model's input

`search("capital of France")`

↓

The result becomes the `Observation:` line

`Observation: The capital of France is Paris.`

The registry is the bridge in the middle of that relay: it is where a word the model wrote becomes an action your code takes.

</div>



**Why keep a list at all?** Two reasons. First, housekeeping: adding a tool means adding one line to one list, and removing a tool means deleting that line. Nothing else in the agent changes. Second, safety, and this is the deeper one: *nothing runs unless it is on the list.* There is no tool to call, only a list to look things up in. If the model invents a tool name, the lookup simply misses, and a miss becomes an Observation telling the model what *is* available. The model can ask for anything. It can only get what is on the list.

The three sub-sections below build this piece by piece: the calculator, the search, and then the registry plus the small function (`dispatch`) that does the looking-up.

### Tool 1: The Calculator

**What this tool is.** A function that takes an arithmetic expression as text, like `"12*34 + 8"`, and returns the answer as text. In the turn, this is what runs when the model's reply says `Action: calculator`.

**Why it is built the careful way.** The obvious way to evaluate a string of math in Python is a built-in called `eval()`, and it is forbidden here, because `eval()` runs *any* code you hand it, including code that deletes files. Remember the trust boundary: the text in `Action Input:` was written by a model, and models can be wrong, confused, or manipulated. So this calculator reads the expression as a structure (using `ast`), checks every piece against a short whitelist of allowed operations (numbers, +, -, *, /, and a few more), and only then does the math. Anything outside the whitelist becomes an error *string*, never a crash and never an execution.

**The cell below** defines the calculator, then tests it on five expressions. Here is what each test represents, and what to expect:

- `"12*34 + 8"`, `"2**10"`, `"(5+3)*2"` — ordinary math. *Expect* the answers: 416, 1024, 16.
- `"__import__('os')"` — a smuggled command, the classic attempt to make the calculator run real code. *Expect* `Error: unsupported expression`. Nothing ran. This line is the whole point of the whitelist.
- `"1/0"` — a math error a model could easily request. *Expect* `Error: division by zero`, handed back as text the model can read and recover from.

**Why this output is your checkpoint:** the last two lines matter more than the first three. They prove the tool fails *safely* — bad input becomes a readable error, not a crash and not a breach — before the loop ever depends on it.

In [5]:
# calculator: a safe arithmetic tool.
# ast.parse turns "12*34" into a tree we can inspect *without running it*.
# We walk the tree and only allow numbers and basic math operators.
# Anything else (try "__import__('os')" below) -> an error *string*, never a crash.
# The model only ever sees the returned string, pasted into an Observation: line.
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv,
       ast.Mod: operator.mod, ast.Pow: operator.pow, ast.USub: operator.neg}

def calculator(expr):
    """Arithmetic via ast -- never eval. Anything outside the whitelist
    becomes an error *string*, not an exception."""
    try:
        tree = ast.parse(expr, mode='eval')
    except SyntaxError:
        return "Error: could not parse expression"
    def _eval(node):
        if isinstance(node, ast.Expression):
            return _eval(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            return OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in OPS:
            return OPS[type(node.op)](_eval(node.operand))
        raise ValueError("unsupported expression")
    try:
        return str(_eval(tree))
    except ZeroDivisionError:
        return "Error: division by zero"
    except (ValueError, TypeError, OverflowError):
        return "Error: unsupported expression"

for expr in ["12*34 + 8", 
             "2**10", "(5+3)*2", 
             "__import__('os')", 
             "1/0"
]:
    print(f"{expr:>14} -> {calculator(expr)}")

     12*34 + 8 -> 416
         2**10 -> 1024
       (5+3)*2 -> 16
__import__('os') -> Error: unsupported expression
           1/0 -> Error: division by zero


### Tool 2: Search

**What this tool is.** A function that takes a question as text and returns an answer as text. In the turn, this is what runs when the model's reply says `Action: search`. In a production agent this would call a real search engine. Here it looks the question up in a tiny built-in index of three facts (`SEARCH_INDEX`), which is enough to teach the shape: question in, answer out, and an honest "no results" when the question is not in the index.

**The cell below** defines the index and the `search` function, then tests it twice. What to expect:

- `search("capital of France")` — a question the index knows. *Expect* the Paris answer. (Notice the function tidies the question first: lowercase, no question mark, so small wording differences still match.)
- `search("how tall is the Eiffel Tower")` — a reasonable question the index does *not* know. *Expect* `No results found...`, said plainly as text. A tool that admits it found nothing is far safer than one that guesses, because the model reads this text and decides what to do next.

**Why this output is your checkpoint:** hit and miss, both behaving. The miss matters most: it proves the tool reports its limits instead of inventing an answer, which is the behavior the whole agent design depends on.

In [6]:
# search: a stand-in for a real search engine.
# SEARCH_INDEX is a tiny built-in index of three facts: question in, canned answer out.
# A question the index does not know gets an honest "No results found" as text,
# never a guess -- the model reads that text and decides what to do next.
SEARCH_INDEX = {
    "boiling point of water in fahrenheit": "The boiling point of water is 212 degrees Fahrenheit.",
    "number of planets": "There are 8 planets in the solar system.",
    "capital of france": "The capital of France is Paris.",
}

def search(query):
    key = query.strip().lower().rstrip('?')
    return SEARCH_INDEX.get(key, f"No results found for '{query}'.")

print(search("capital of France"))              # a hit: the Paris answer
print(search("how tall is the Eiffel Tower"))   # a miss: honest "No results found"

The capital of France is Paris.
No results found for 'how tall is the Eiffel Tower'.


### The Registry and Dispatch: Putting the Tools to Work

Two pieces remain in this section, and now they can be defined in one breath each:

- **The registry** (`TOOLS`) is the list from the section opening, written as code: each tool's name paired with its function. Two lines long. That really is the whole registry.
- **Dispatch** (`dispatch`) is the small function that uses the registry. It takes the name and input the parser pulled out of a reply, looks the name up, runs the matching tool, and wraps whatever comes back as an `Observation:` line. Dispatch is the only road from a parsed reply to a running tool.

**The cell below** defines both, then runs dispatch three times. These three calls are the checkpoint for the entire tools section:

1. `dispatch("calculator", "12*34")` — a name that is on the list, with good input. *Expect:* `Observation: 408`. The relay worked end to end: name → lookup → run → wrapped result.
2. `dispatch("search", "capital of France")` — the other tool, same road. *Expect:* an Observation carrying the Paris answer.
3. `dispatch("web_search", "cats")` — a name that is **not** on the list. Models do this constantly: they ask for the tool they *expect* to exist. *Expect:* an Observation saying the tool is unknown and listing what *is* available. No crash, no exception escaping. The loop will feed that Observation back to the model, and the model gets to try again with a real tool.

**Why this output is your checkpoint:** one line per path — hit, hit, safe miss. Once you have seen all three behave, the parser plus dispatch plus registry chain is proven, and the loop in the next section can build on it without re-checking.

In [7]:
# TOOLS + dispatch: the registry, and the only road from a name to a running tool.
# TOOLS is the registry -- a plain dict mapping names to functions.
# dispatch() looks up the name the model wrote. Unknown name? That becomes an
# Observation: line telling the model what *is* available, and the loop continues.
# A tool that errors? Also an Observation:. The loop never crashes on a bad action.
TOOLS = {"calculator": calculator, "search": search}

def dispatch(action_name, action_input):
    """Route a parsed action to a tool. Misses and errors become
    observations -- the loop never crashes on a bad action."""
    if action_name not in TOOLS:
        return (f"Observation: unknown tool '{action_name}'. "
                f"Available tools: calculator, search.")
    try:
        result = TOOLS[action_name](action_input)
    except Exception as e:
        return f"Observation: tool '{action_name}' raised {type(e).__name__}: {e}."
    return f"Observation: {result}"

print(dispatch("calculator", "12*34"))
print(dispatch("search", "capital of France"))
print(dispatch("web_search", "cats"))

Observation: 408
Observation: The capital of France is Paris.
Observation: unknown tool 'web_search'. Available tools: calculator, search.


## The Loop: Everything, Repeating

**Definition.** The *loop* is the piece of code that runs the turns. It holds the prompt, hands it to the model, reads the reply, acts on it, writes the result into the prompt, and goes around again. If the contract is the agreement and the tools are the hands, the loop is the conductor: five steps, repeated.

1. Ask the model, using the current prompt.
2. If the reply says `Final Answer:`, stop. That is the answer.
3. Otherwise, parse the reply (no action found: the Observation is a format correction).
4. Dispatch the action; the result becomes the Observation.
5. Append the reply and the Observation to the prompt. Go to 1. (Safety cap: 8 turns, so a confused model cannot loop forever.)

**But first, what is the "model" the loop asks?** In your real Week 1 build, it is a function you wrote on Day 1 called `ask()`, in a file named `ask.py`. Its whole job: take a prompt as text, send it to a real model over the internet (Gemini, in your build), and return the model's reply as text. Text in, text out. Later you wrote `ask_gateway()`, which does the identical job through the work gateway instead. Same shape, different supplier. *That* function, one prompt in and one reply out, is what the loop calls once per turn.

**In this notebook, a stand-in plays that role**, so the lesson needs no key, no internet, and no money, and gives the same result every run. Here is the contrast, side by side:

| | Your real `ask()` | The notebook's `MockModel` |
|---|---|---|
| Where replies come from | A real model, live, over the internet | A list of replies written in advance, popped off one per turn |
| Same question twice | Different wording every time | Identical, every time |
| Needs | An API key, internet, a budget | Nothing |
| Why it exists | Production: real answers | Teaching: every failure is reproducible |

The swap works because of a fact worth pausing on: **the loop cannot tell the difference.** It never sees a model. It calls something with a prompt and gets text back. Where the text came from is not the loop's business. That is not a teaching simplification. That is the architecture, and it is why you will be able to swap models in later weeks without rebuilding anything.

**When you run the cell below,** expect three things, in this order. First, for each turn, the *full prompt exactly as the model receives it*, printed under a "What the model sees" header. Read the bottom of each one: that is where the newest reply and Observation were glued on, and it is the only place the agent's "memory" lives. Second, after the run, a tidy trace of the turns. Third, the prompt's length in characters after each turn. The lengths grow every single turn. Notice that. Week 3 is entirely about that problem.

> **Simulated model ahead.** The next cells use `MockModel`, a stand-in that replays pre-written replies instead of calling a real API. Everything the loop does with those replies (parse, dispatch, recover from garbage) is identical to production. Only the *source* of the text is fake, which is what makes every demo reproducible.

In [8]:
# MockModel + run(): the whole agent, assembled.
# MockModel is a SIMULATED model -- not a real API call. It pops pre-written
# replies from a list, one per turn, like an actor reading from a script.
# run() is the loop: ask, parse, dispatch, append the observation, repeat.
# It prints the full prompt each turn, so you can watch the only "memory"
# the agent has grow at the bottom, one reply + Observation at a time.
class MockModel:
    """Deterministic stand-in for ask(): pops canned replies in order.
    The loop cannot tell the difference -- it only ever sees text."""

    def __init__(self, script):
        self.script = list(script)
        self.calls = 0

    def __call__(self, prompt):
        self.calls += 1
        if not self.script:
            return "Thought: I am out of scripted replies.\nFinal Answer: unknown"
        return self.script.pop(0)

# run(): the whole agent in one function. Five lines of logic wearing a trench coat:
#   1. ask the model (here, the mock) with the current prompt
#   2. if it wrote "Final Answer:", stop and return the answer
#   3. parse the Action: line (None -> append a format-correction observation)
#   4. dispatch the tool, wrap the result as an Observation: line
#   5. append reply + observation to the prompt, repeat (safety cap: 8 turns)
def run(question, model, max_iterations=8, track_prompt_len=False, show_prompt=False):
    """The whole agent: ask, parse, dispatch, append observation, repeat.
    show_prompt prints the exact prompt text the model receives each turn."""
    prompt = build_prompt(question)
    trace, prompt_lens = [], []
    for turn in range(1, max_iterations + 1):
        if track_prompt_len:
            prompt_lens.append(len(prompt))
        if show_prompt:
            print(f"===== What the model sees on turn {turn} =====")
            print(prompt)
            print("===== end of prompt =====\n")
        reply = model(prompt)
        if "Final Answer:" in reply:
            answer = reply.split("Final Answer:")[1].strip()
            trace.append({"answer": answer})
            return {"answer": answer, "trace": trace, "prompt_lens": prompt_lens}
        parsed = parse_action(reply)
        if parsed is None:
            obs = ("Observation: I couldn't parse an action from your last response. "
                   "Reply using exactly the format: Thought: ... Action: ... Action Input: ...")
            trace.append({"raw": reply, "observation": obs})
        else:
            name, action_input = parsed
            obs = dispatch(name, action_input)
            thought = re.search(r'Thought:\s*(.+)', reply)
            trace.append({"thought": thought.group(1).strip() if thought else "",
                          "action": f"{name}({action_input})",
                          "observation": obs})
        prompt += "\n" + reply + "\n" + obs + "\n"
    return {"answer": "max iterations reached", "trace": trace, "prompt_lens": prompt_lens}


def print_trace(result):
    for i, t in enumerate(result["trace"], 1):
        if "answer" in t:
            print(f"--- Turn {i}: FINAL ---\nFinal Answer: {t['answer']}\n")
        elif "raw" in t:
            print(f"--- Turn {i}: MALFORMED ---\nRaw reply: {t['raw']}\n{t['observation']}\n")
        else:
            print(f"--- Turn {i} ---\nThought: {t['thought']}\n"
                  f"Action: {t['action']}\n{t['observation']}\n")
    print("Answer:", result["answer"])

script = [
    "Thought: I need the boiling point of water in Fahrenheit.\nAction: search\nAction Input: boiling point of water in Fahrenheit",
    "Thought: Now I need the number of planets.\nAction: search\nAction Input: number of planets",
    "Thought: Now multiply 212 by 8.\nAction: calculator\nAction Input: 212*8",
    "Thought: I have everything I need.\nFinal Answer: 1696",
]

result = run(
    "What is the boiling point of water in Fahrenheit, multiplied by the number of planets?",
    MockModel(script),
    track_prompt_len=True,
    show_prompt=True,
)
print_trace(result)
print("Prompt length per iteration:", result["prompt_lens"])

===== What the model sees on turn 1 =====
You solve problems by reasoning step by step, then taking actions.
Reply using exactly the format below, nothing else.

Thought: <your reasoning>
Action: <one of: calculator, search>
Action Input: <input for the action>
... (repeat Thought/Action/Action Input as needed)
Thought: <final reasoning>
Final Answer: <the answer>

Example:
Question: What is 12*34 plus 8?
Thought: I need to compute 12*34 first.
Action: calculator
Action Input: 12*34
Observation: 408
Thought: Now add 8.
Action: calculator
Action Input: 408+8
Observation: 416
Thought: I have the answer.
Final Answer: 416

Question: What is the boiling point of water in Fahrenheit, multiplied by the number of planets?

===== end of prompt =====

===== What the model sees on turn 2 =====
You solve problems by reasoning step by step, then taking actions.
Reply using exactly the format below, nothing else.

Thought: <your reasoning>
Action: <one of: calculator, search>
Action Input: <input f

## Going Live: Your Build

Everything above ran on a simulated model. Your Week 1 build ran on the real thing. Here is the entire difference.

The notebook's loop takes the model as an argument: `run(question, model, max_iterations=8)`. Your `ask_gateway(prompt)` is the same shape as the mock: text in, text out. So going live is one argument:

```python
from ask_gateway import ask_gateway   # your real model call (needs your key)

run("What is the boiling point of water in Fahrenheit, "
    "multiplied by the number of planets?",
    model=ask_gateway)                # <- the only change. Run this on your laptop.
```

The loop never knew the difference, because the loop only ever sees text. That is not a simplification for teaching. That is the architecture.

### Where each piece lives in your repo

| Notebook section | Your file | Day |
|---|---|---|
| `parse_action` | `week-01-react-loop/react.py` | Day 2 |
| `calculator`, `search`, registry | `week-01-react-loop/tools.py` (`run_tool`) | Day 3 |
| `run()` | `week-01-react-loop/agent.py` | Day 4 |
| The three breakages | `week-01-react-loop/day-05-breakage-log.md` | Day 5 |

### Spot the difference (exercise)

Your real `agent.py` has two things the notebook's `run()` does not. For each one, explain out loud what breaks without it:

1. **The `tools_used` counter and the forced-tool rule.** A `Final Answer:` with zero tool calls gets bounced back with a correction instead of accepted. Why would the notebook's loop accept it?
2. **The prompt reset on a premature `Final Answer:`.** The prompt is rebuilt from the question plus the correction, so the model never sees its own rejected answer in the history. What goes wrong if you just append the correction and continue?

If you can narrate both, you understand your own build better than the notebook does. That is the point.

## Your Build, Day by Day

The lesson is above. This is the build it was teaching toward: five days, one piece per day, each piece checked before the next one sits on top of it. The reference code for every piece is in the next section, heavily commented.

| Day | You build | In file | Done when |
|---|---|---|---|
| 1 | `ask()` — a reusable function that sends a prompt to a real model and returns its reply text | `ask.py` | Running it prints a real model's reply, with no manual steps |
| 2 | The contract — `build_prompt()` and `parse_action()` | `react.py` | The parser returns `('calculator', '12*34 + 8')` for a clean reply, `None` for a reply with no action, and still works through markdown fences |
| 3 | The tools — `calculator`, `search`, and the registry (`run_tool`) | `tools.py` | Both tools answer through the registry, and an unknown tool name returns an "unknown tool" message instead of crashing |
| 4 | The loop — `run()`, wiring ask + prompt + parser + tools together | `agent.py` | Three test questions get answered by actually using the tools, with a trace you can read |
| 5 | Prove it — break your own agent on purpose and write down what happened | `day-05-breakage-log.md` | The log covers a malformed reply, an unknown tool, and the iteration cap, in your own words |

**Day 1 — `ask()`.** Before an agent can loop, it needs a voice. One function: prompt in, the model's reply text out. The API key comes from an environment variable, never written in the code. Nothing else today: no tools, no loop, no parsing. Done when `python ask.py` prints a Gemini reply end to end.

**Day 2 — the contract.** The prompt template (`build_prompt`) that slots a question into the ReAct format, and the parser (`parse_action`) that pulls the action back out. Done when the three parser checks in this notebook's parser section pass against *your* parser, not the notebook's.

**Day 3 — the tools.** The safe calculator, the search stub, and the registry that routes a name to a function (`run_tool` in your build; the notebook calls the same job `dispatch`). Finish the day by running one full turn by hand: build prompt → ask → parse → run tool → read the Observation. No loop yet. The hand-run turn is the checkpoint that every piece works before the loop automates them.

**Day 4 — the loop.** `run()`: ask, check for Final Answer, parse, dispatch, append reply + Observation, repeat, with an iteration cap so a confused model cannot spin forever. Done when three questions (at least one needing each tool, one needing both) complete with readable traces.

**Day 5 — prove it.** Run the three breakages against the real model, not the mock: a malformed reply, an unknown tool, a question designed to hit the iteration cap. Log what the mock did, what reality did, and the finding. Then the explain test, out loud: narrate all four paths (normal turn, malformed, unknown tool, cap) with no notes. The log section is at the end of this notebook.

## Reference Code: The Build, File by File

Below is the Week 1 build as four files, the way it lives in your repo. This is reference, not a copy source: read it to understand the shape, then type your own version in your own words and comments. Each file gets a walkthrough first — what its functions are, where they fit in the loop, why they exist — and then the code itself, commented heavily on purpose.

A naming bridge, so the two vocabularies do not confuse you: the notebook's `dispatch` is your build's `run_tool`. Same job (name in, Observation out), different name. Everything else matches.

### Reference: `ask.py` (Day 1)

**Walkthrough.** `ask(prompt)` is the agent's only connection to a model. It takes the prompt as one string, packages it the way the Gemini API expects, sends it over the internet with your key, digs the reply text out of the response, and returns that string. Every other file treats this function as a black box: text in, text out. That is exactly the shape the loop needs, and exactly the shape `MockModel` imitates in this notebook, and exactly the shape `ask_gateway()` later reused through the work gateway. Notice what `ask()` does *not* do: no parsing, no tools, no opinions about the reply. One job.

```python
"""Day 1 - a reusable Gemini wrapper.

No tools, no loops, no parsing yet. One job: send a prompt, get text back.
The API key comes from the GEMINI_API_KEY environment variable, never from code.
"""

import json
import os
import urllib.request

# Which model we are talking to, and where it lives on the internet.
# Keeping these at the top means changing models later is a one-line edit.
MODEL = "gemini-3.8-flash"
ENDPOINT = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"


def ask(prompt: str) -> str:
    """Send *prompt* to Gemini and return the model's text reply."""
    # The key comes from the environment (set once in the terminal),
    # so it never appears in code, in git, or in a screenshot.
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise RuntimeError("GEMINI_API_KEY environment variable is not set")

    # Package the prompt the way the API expects: a JSON body with one
    # "contents" entry holding the text. This shape is the API's contract.
    body = json.dumps({"contents": [{"parts": [{"text": prompt}]}]}).encode("utf-8")

    # Build the HTTP request: the URL carries the key, the header says
    # "the body is JSON", the method says "we are sending data" (POST).
    request = urllib.request.Request(
        f"{ENDPOINT}?key={api_key}",
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    # Send it and read the whole reply. This is the one moment per call
    # where the program waits on the internet.
    with urllib.request.urlopen(request) as response:
        payload = json.loads(response.read().decode("utf-8"))

    # The reply text is buried in the response structure. Dig it out and
    # return just the string: callers of ask() should never see the JSON.
    return payload["candidates"][0]["content"]["parts"][0]["text"]
```

### Reference: `react.py` (Day 2)

**Walkthrough.** This file holds the contract, both halves. `build_prompt(question)` is the *promise* half: it slots a question into `REACT_PROMPT`, the template that tells the model the exact reply format and shows it a worked example. `parse_action(reply)` is the *verification* half: two regexes that pull the action name and its input out of a reply, returning `None` when there is no action to pull. Inside the loop, this file is used twice per turn: `build_prompt` once, at the very start, and `parse_action` on every reply the model sends back.

````python
"""Day 2 - the ReAct contract: the prompt template and the parser."""

import re

# The contract, written in plain English for the model.
# It fixes the reply format, names the allowed tools, and shows ONE worked
# example so the shape is unambiguous. The {question} slot is where
# build_prompt() drops the actual question, at the very end, so the model
# reads the instructions first and the job last.
REACT_PROMPT = """You solve problems by reasoning step by step, then taking actions.
Reply using exactly the format below, nothing else.

Thought: <your reasoning>
Action: <one of: calculator, search>
Action Input: <input for the action>
... (repeat Thought/Action/Action Input as needed)
Thought: <final reasoning>
Final Answer: <the answer>

Example:
Question: What is 12*34 plus 8?
Thought: I need to compute 12*34 first.
Action: calculator
Action Input: 12*34
Observation: 408
Thought: Now add 8.
Action: calculator
Action Input: 408+8
Observation: 416
Thought: I have the answer.
Final Answer: 416

Question: {question}
"""


def build_prompt(question):
    """Slot *question* into the contract template and return the full prompt."""
    return REACT_PROMPT.format(question=question)


def parse_action(reply):
    """Pull (action_name, action_input) out of a model reply.

    Returns None when the reply has no Action: line. None is a signal
    ("the model broke the format"), never a crash: the loop turns it into
    a correction Observation and the model gets another turn.
    """
    text = reply.strip()
    # Models love wrapping replies in markdown fences. Peel them off first
    # so the searches below see the same text either way.
    text = re.sub(r'^```[a-z]*\n?', '', text)   # opening fence
    text = re.sub(r'\n?```$', '', text)          # closing fence

    # First regex: find "Action:", skip any spaces, capture the tool's name.
    # The parentheses are the part we keep; group(1) is that capture.
    m_action = re.search(r'Action:\s*(\w+)', text)
    if not m_action:
        return None  # no action in this reply: signal, not an error

    # Second regex: find "Action Input:", capture everything to the end
    # of the line. A missing input line is allowed and means "".
    m_input = re.search(r'Action Input:\s*(.+)', text)
    return m_action.group(1), (m_input.group(1).strip() if m_input else '')
````

### Reference: `tools.py` (Day 3)

**Walkthrough.** Three things live here. `calculator(expr)` does arithmetic safely (structure check first, whitelist only, errors returned as strings). `search(query)` looks a question up in a small canned index and honestly reports a miss. `run_tool(name, input)` is the registry plus dispatch in one function: it looks the name up in `TOOLS`, runs the match, and returns the result wrapped as an `Observation:` line; an unknown name or a tool error also comes back as an Observation, never as a crash. Inside the loop, this file is touched exactly once per turn, right after the parser, and the string it returns is what gets glued onto the prompt.

```python
"""Day 3 - the tools and the registry that routes to them."""

import ast
import operator

# --- calculator -------------------------------------------------------------
# The whitelist: the only operations this tool will ever perform.
# ast.Add is the *kind* of node, operator.add is the *function* that does it.
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv,
       ast.Mod: operator.mod, ast.Pow: operator.pow, ast.USub: operator.neg}


def calculator(expr):
    """Evaluate an arithmetic string safely. Never eval().

    The expression is parsed into a tree and every node is checked
    against the whitelist before anything is computed. Anything else --
    a smuggled function call, a name, a string -- becomes an error
    *string* the model can read, not an exception and not an execution.
    """
    try:
        tree = ast.parse(expr, mode='eval')
    except SyntaxError:
        return "Error: could not parse expression"

    def _eval(node):
        # Walk the tree by hand. Only three shapes are allowed:
        if isinstance(node, ast.Expression):   # the wrapper node: look inside
            return _eval(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value                  # a plain number: allowed
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            # a binary operation whose operator is whitelisted: do it
            return OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in OPS:
            return OPS[type(node.op)](_eval(node.operand))  # e.g. a minus sign
        raise ValueError("unsupported expression")  # anything else: refuse

    try:
        return str(_eval(tree))
    except ZeroDivisionError:
        return "Error: division by zero"
    except (ValueError, TypeError, OverflowError):
        return "Error: unsupported expression"


# --- search -----------------------------------------------------------------
# A stub standing in for a real search engine: a tiny fixed index.
# The shape is the lesson -- question in, answer (or honest miss) out.
SEARCH_INDEX = {
    "boiling point of water in fahrenheit": "The boiling point of water is 212 degrees Fahrenheit.",
    "number of planets": "There are 8 planets in the solar system.",
    "capital of france": "The capital of France is Paris.",
}


def search(query):
    """Look *query* up in the index. A miss is reported, never guessed."""
    key = query.strip().lower().rstrip('?')  # tidy so small differences still match
    return SEARCH_INDEX.get(key, f"No results found for '{query}'.")


# --- the registry + dispatch -------------------------------------------------
# THE REGISTRY. Name -> function. This dict is the complete list of what
# the agent is allowed to do. Adding a tool = adding one line here.
TOOLS = {"calculator": calculator, "search": search}


def run_tool(action_name, action_input):
    """Run the tool named *action_name* and return an Observation: string.

    Unknown name -> an Observation saying so and listing what exists.
    Tool raised -> an Observation carrying the error. Neither crashes:
    the loop feeds the Observation back and the model gets to recover.
    """
    if action_name not in TOOLS:
        return (f"Observation: unknown tool '{action_name}'. "
                f"Available tools: calculator, search.")
    try:
        result = TOOLS[action_name](action_input)
    except Exception as e:
        return f"Observation: tool '{action_name}' raised {type(e).__name__}: {e}."
    return f"Observation: {result}"
```

### Reference: `agent.py` (Day 4)

**Walkthrough.** This is the conductor, and it is short because every hard job was given to a specialist: `ask()` talks to the model, `build_prompt` writes the contract, `parse_action` reads replies, `run_tool` runs tools. `run(question)` just repeats the relay: build the prompt once, then per turn — ask, stop if the reply carries a Final Answer, parse, run the tool, glue reply + Observation onto the prompt. The iteration cap (8) is the seatbelt: a confused model gets eight turns, not infinity. Run from the command line, it asks the three test questions from the Day 4 sheet. (Your hardened version adds the `tools_used` counter and the prompt reset from the spot-the-difference exercise; those came out of Day 5's breakage findings, and the plain loop below is what they were added to.)

```python
"""Day 4 - the loop: ask, parse, dispatch, append, repeat."""

from ask import ask
from react import build_prompt, parse_action
from tools import run_tool

# The seatbelt. A model that never produces a Final Answer gets this
# many turns and no more, so a run always ends.
MAX_ITERATIONS = 8


def run(question):
    """Run the ReAct loop on *question* and return the final answer."""
    # The prompt starts as the contract with our question slotted in.
    # From here on, the prompt IS the conversation: everything that
    # happens gets appended to it, and it is all the model ever sees.
    prompt = build_prompt(question)

    for turn in range(1, MAX_ITERATIONS + 1):
        # 1. ASK: send the whole current prompt, get one reply back.
        reply = ask(prompt)
        print(f"--- turn {turn} ---")
        print(reply)

        # 2. STOP?: a Final Answer ends the loop. Everything after the
        # "Final Answer:" marker is the answer.
        if "Final Answer:" in reply:
            return reply.split("Final Answer:")[1].strip()

        # 3. PARSE: pull the action out of the reply. None means the
        # model broke the format: the Observation is a correction that
        # tells it the required shape, and the loop continues.
        parsed = parse_action(reply)
        if parsed is None:
            obs = ("Observation: I couldn't parse an action from your last response. "
                   "Reply using exactly the format: Thought: ... Action: ... Action Input: ...")
        else:
            # 4. DISPATCH: run the tool through the registry. Whatever
            # comes back (result, unknown tool, error) is already an
            # Observation: string, ready to paste.
            name, action_input = parsed
            obs = run_tool(name, action_input)
        print(obs)

        # 5. APPEND: glue the reply and the Observation onto the prompt.
        # This growing text is the agent's only memory. Next turn, the
        # model reads the whole history and picks up where it left off.
        prompt += "\n" + reply + "\n" + obs + "\n"

    return "max iterations reached"  # the cap fired: report it, don't hide it


if __name__ == "__main__":
    # The Day 4 test questions: one per tool, one needing both.
    print(run("What is the capital of France?"))
    print(run("What is 12*34 plus 8?"))
    print(run("What is the boiling point of water in Fahrenheit, "
              "multiplied by the number of planets?"))
```

## Prove It: Mock vs Reality

The notebook proved the loop's logic against a script. Now prove it against the one thing the mock cannot simulate: a real model.

**1. Run your real build.** In your repo, run your Week 1 `agent.py` against the boiling-point question (or three fresh ones needing both tools). Save the trace.

**2. Replay the breakages live.** Feed your real loop the same three attacks from the worked example: a malformed reply, an unknown tool, an unanswerable question. Record what happens.

**3. Write the comparison below.** For each case, answer one question: where did reality disagree with the mock? The mock always recovers politely; the real model may loop, ramble, or surprise you. Every disagreement is a finding. This section doubles as the week's breakage log: fill it in Friday, in your own words.

**4. Rehearse the explain test.** Out loud, no notes: the normal turn, the malformed call, the unknown tool, the iteration cap. If any path is fuzzy, re-read that day's trace until it's crisp.

The mock tells you the loop is correct. Reality tells you it is robust. You need both before the week is done.

## Breakage Log: Mock vs Reality

_Fill this in Friday, in your own words._

### Case 1: malformed reply
- **Mock did:**
- **Reality did:**
- **Finding:**

### Case 2: unknown tool
- **Mock did:**
- **Reality did:**
- **Finding:**

### Case 3: iteration cap
- **Mock did:**
- **Reality did:**
- **Finding:**

### Explain test rehearsal (out loud, no notes)
- [ ] normal two-step turn
- [ ] malformed tool call
- [ ] unknown tool
- [ ] iteration cap
- **Fuzzy spots to re-read:**

### Traces
_Paste the saved trace files from today's live runs here, or note their paths in the repo. Week 6's eval harness will eat these._

## What You Own Now

- The model emits text; your code acts. There is no magic step.
- Observations are just prompt text. The context grows every iteration.
- Error handling *is* the agent. The loop that recovers is the whole game.

Your Week 1 build was the exercise for this lesson: `ask()`, the ReAct prompt, the parser, the tools, the loop, three questions, and the breakage log. If you can narrate all four explain-test paths out loud right now, the notebook did its job.

**Where this goes:** the trace format you printed above is the raw material for the Week 6 eval harness. And the ever-growing prompt is the problem Week 3 exists to solve.